In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


In [2]:
import os

In [3]:
os.listdir(path)

['chest_xray']

In [4]:
os.listdir(path + "/chest_xray")

['chest_xray', '__MACOSX', 'val', 'test', 'train']

In [5]:
train_path = path + "/chest_xray/train"
test_path = path + "/chest_xray/test"
val_path = path + "/chest_xray/val"

In [8]:
print(train_path)
print(test_path)
print(val_path)

/kaggle/input/chest-xray-pneumonia/chest_xray/train
/kaggle/input/chest-xray-pneumonia/chest_xray/test
/kaggle/input/chest-xray-pneumonia/chest_xray/val


In [10]:
print(os.listdir(train_path))
print(os.listdir(test_path))
print(os.listdir(val_path))

['PNEUMONIA', 'NORMAL']
['PNEUMONIA', 'NORMAL']
['PNEUMONIA', 'NORMAL']


In [17]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

In [19]:
train_data = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(180, 180),
    subset = "training",
    validation_split=0.2,
    seed = 42,
    batch_size=32,
    shuffle=True,
)

Found 5216 files belonging to 2 classes.
Using 4173 files for training.


In [20]:
test_data = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(180, 180),
    batch_size=32,
    validation_split=0.2,
    subset="validation",
    seed=42,
    shuffle=True,
)

Found 624 files belonging to 2 classes.
Using 124 files for validation.


In [21]:
valid_data = tf.keras.utils.image_dataset_from_directory(
    val_path,
    image_size=(180, 180),
    batch_size=32,
    subset="validation",
    validation_split=0.2,
    seed=42,
    shuffle=True,
)


Found 16 files belonging to 2 classes.
Using 3 files for validation.


In [23]:
model = keras.Sequential([
    layers.Input(shape=(180, 180, 3)),
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dropout(0.5),  # overfitting kam karne ke liye
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

In [24]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [25]:
model.fit(train_data, epochs=10, validation_data=valid_data)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 45s 288ms/step - accuracy: 0.8243 - loss: 0.4136 - val_accuracy: 0.6667 - val_loss: 0.3247
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 26s 196ms/step - accuracy: 0.9487 - loss: 0.1369 - val_accuracy: 0.6667 - val_loss: 0.5447
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 40s 192ms/step - accuracy: 0.9573 - loss: 0.1146 - val_accuracy: 1.0000 - val_loss: 0.0978
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 26s 195ms/step - accuracy: 0.9732 - loss: 0.0811 - val_accuracy: 1.0000 - val_loss: 0.0683
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 25s 192ms/step - accuracy: 0.9724 - loss: 0.0768 - val_accuracy: 1.0000 - val_loss: 0.1122
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 25s 192ms/step - accuracy: 0.9763 - loss: 0.0633 - val_accuracy: 1.0000 - val_loss: 0.0708
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 41s 191ms/step - accuracy: 0.9787 - loss: 0.0546 - val_accuracy: 1.0000 - val_loss: 0.0337
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 25s 192ms/step - accuracy: 0.9794 - loss: 0

In [26]:
model.evaluate(test_data)

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 349ms/step - accuracy: 0.7177 - loss: 2.4030


[2.4029757976531982, 0.7177419066429138]

In [32]:
model.save("pneumonia_detector.keras")

In [29]:
import gradio as gr
import numpy as np
from PIL import Image

def predict_pneumonia(img):
    img = img.resize((180, 180))
    img_array = np.array(img)


    if len(img_array.shape) == 2:
        img_array = np.stack((img_array,)*3, axis=-1)


    img_array = np.expand_dims(img_array, axis=0)


    prediction = model.predict(img_array)[0][0]

    if prediction > 0.5:
        confidence = prediction * 100
        return f"PNEUMONIA detected ({confidence:.2f}% confidence)"
    else:
        confidence = (1 - prediction) * 100
        return f"NORMAL ({confidence:.2f}% confidence)"

interface = gr.Interface(
    fn=predict_pneumonia,
    inputs=gr.Image(type="pil"),
    outputs=gr.Text(),
    title="Normal & Pneumonia X-Ray Detector",
    description="Chest X-Ray image upload karo, model batayega Normal hai ya Pneumonia."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://db0bf9e024eef7821f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
